In [ ]:
import json
import numpy as np
from datasets import Dataset
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments
import torch
from torch.utils.data import DataLoader
from vncorenlp import VnCoreNLP
from imblearn.over_sampling import RandomOverSampler

#VnCoreNLP
vncorenlp = VnCoreNLP("/kaggle/input/vncorenlp/VnCoreNLP-1.2/VnCoreNLP-1.1.1.jar", annotators="wseg", max_heap_size='-Xmx2g')

def load_data(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    label_map = {"non-sarcasm": 0, "sarcasm": 1}
    texts = [item["caption"] for item in data]
    labels = [label_map[item["label"].strip().lower()] for item in data]
    return texts, labels

def word_segmentation(texts):
    segmented_texts = []
    for text in texts:
        segmented = vncorenlp.tokenize(text)
        segmented_text = " ".join([" ".join(sentence) for sentence in segmented])
        segmented_texts.append(segmented_text)
    return segmented_texts

def preprocess_dataset(texts, labels, tokenizer):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=128)
    dataset = Dataset.from_dict({
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "labels": labels
    })
    return dataset


def train_with_early_stopping(model, train_dataset, val_dataset, training_args, patience=2):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=training_args.learning_rate)

    train_loader = DataLoader(train_dataset, batch_size=training_args.per_device_train_batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=training_args.per_device_eval_batch_size)

    best_f1 = 0.0
    patience_counter = 0

    for epoch in range(int(training_args.num_train_epochs)):
        model.train()
        total_loss = 0.0

        for batch in train_loader:
            inputs = {
                "input_ids": torch.tensor(batch["input_ids"]).to(device),
                "attention_mask": torch.tensor(batch["attention_mask"]).to(device),
                "labels": torch.tensor(batch["labels"]).to(device),
            }

            optimizer.zero_grad()
            outputs = model(**inputs)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"\nEpoch {epoch+1}: Train loss = {avg_loss:.4f}")

        model.eval()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for batch in val_loader:
                inputs = {
                    "input_ids": torch.tensor(batch["input_ids"]).to(device),
                    "attention_mask": torch.tensor(batch["attention_mask"]).to(device),
                }
                labels = torch.tensor(batch["labels"]).to(device)

                outputs = model(**inputs)
                preds = torch.argmax(outputs.logits, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        acc = accuracy_score(all_labels, all_preds)
        prec = precision_score(all_labels, all_preds, average="macro")
        rec = recall_score(all_labels, all_preds, average="macro")
        f1 = f1_score(all_labels, all_preds, average="macro")

        print(f"Val Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            torch.save(model.state_dict(), "best_phobert_model.pt")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break

def evaluate_on_test(model, test_dataset, batch_size=32):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            inputs = {
                "input_ids": torch.tensor(batch["input_ids"]).to(device),
                "attention_mask": torch.tensor(batch["attention_mask"]).to(device),
            }
            labels = torch.tensor(batch["labels"]).to(device)
            outputs = model(**inputs)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average="macro")
    rec = recall_score(all_labels, all_preds, average="macro")
    f1 = f1_score(all_labels, all_preds, average="macro")

    print("\n===== Test Set Evaluation =====")
    print(f"Test Accuracy: {acc:.4f}")
    print(f"Test Precision (macro): {prec:.4f}")
    print(f"Test Recall (macro): {rec:.4f}")
    print(f"Test F1 (macro): {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=["Non-sarcasm", "Sarcasm"], digits=4))

def main():
    llm_train_path = "/kaggle/input/phobertdata/silver_text.json"   
    human_train_path = "/kaggle/input/phobertdata/text_train.json"  
    val_path = "/kaggle/input/phobertdata/text_dev.json"
    test_path = "/kaggle/input/phobertdata/text_test.json"

    llm_texts, llm_labels = load_data(llm_train_path)
    human_texts, human_labels = load_data(human_train_path)

    # Word segmentation
    llm_texts = word_segmentation(llm_texts)
    human_texts = word_segmentation(human_texts)

    #Oversample
    def oversample_text_label(texts, labels):
        texts_np = np.array(texts).reshape(-1, 1)
        ros = RandomOverSampler(sampling_strategy="auto")
        texts_resampled, labels_resampled = ros.fit_resample(texts_np, labels)
        return texts_resampled.ravel().tolist(), labels_resampled

    human_texts_os, human_labels_os = oversample_text_label(human_texts, human_labels)

    train_texts = llm_texts + human_texts_os
    train_labels = llm_labels + human_labels_os

    val_texts, val_labels = load_data(val_path)
    test_texts, test_labels = load_data(test_path)
    val_texts = word_segmentation(val_texts)
    test_texts = word_segmentation(test_texts)

    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
    model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base", num_labels=2)

    train_dataset = preprocess_dataset(train_texts, train_labels, tokenizer)
    val_dataset = preprocess_dataset(val_texts, val_labels, tokenizer)
    test_dataset = preprocess_dataset(test_texts, test_labels, tokenizer)

    train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    training_args = TrainingArguments(
        output_dir="./results",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=8,
        weight_decay=0.01,
        logging_dir="./logs",
        logging_steps=10,
        report_to="none"
    )

    train_with_early_stopping(model, train_dataset, val_dataset, training_args, patience=2)

    model.load_state_dict(torch.load("best_phobert_model.pt"))
    evaluate_on_test(model, test_dataset)
    
    vncorenlp.close()

if __name__ == "__main__":
    main()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_103/2007835277.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(batch["input_ids"]).to(device),
/tmp/ipykernel_103/2007835277.py:65: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(batch["attention_mask"]).to(device),
/tmp/ipykernel_103/2007835277.py:66: 


Epoch 1: Train loss = 0.5840


/tmp/ipykernel_103/2007835277.py:86: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(batch["input_ids"]).to(device),
/tmp/ipykernel_103/2007835277.py:87: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(batch["attention_mask"]).to(device),
/tmp/ipykernel_103/2007835277.py:89: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(batch["labels"]).to(device)


Val Accuracy: 0.5650 | Precision: 0.5247 | Recall: 0.6289 | F1: 0.4238


/tmp/ipykernel_103/2007835277.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(batch["input_ids"]).to(device),
/tmp/ipykernel_103/2007835277.py:65: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(batch["attention_mask"]).to(device),
/tmp/ipykernel_103/2007835277.py:66: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "labels": torch.tensor(batch["labels"]).to(device),


...


Epoch 6: Train loss = 0.1267


/tmp/ipykernel_103/2007835277.py:86: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(batch["input_ids"]).to(device),
/tmp/ipykernel_103/2007835277.py:87: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(batch["attention_mask"]).to(device),
/tmp/ipykernel_103/2007835277.py:89: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(batch["labels"]).to(device)


Val Accuracy: 0.7200 | Precision: 0.5376 | Recall: 0.6632 | F1: 0.5039
Early stopping triggered at epoch 6


/tmp/ipykernel_103/2007835277.py:126: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(batch["input_ids"]).to(device),
/tmp/ipykernel_103/2007835277.py:127: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(batch["attention_mask"]).to(device),
/tmp/ipykernel_103/2007835277.py:129: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(batch["labels"]).to(device)



===== Test Set Evaluation =====
Test Accuracy: 0.7600
Test Precision (macro): 0.5356
Test Recall (macro): 0.6368
Test F1 (macro): 0.5160

Classification Report:
              precision    recall  f1-score   support

 Non-sarcasm     0.9671    0.7737    0.8596       190
     Sarcasm     0.1042    0.5000    0.1724        10

    accuracy                         0.7600       200
   macro avg     0.5356    0.6368    0.5160       200
weighted avg     0.9240    0.7600    0.8253       200

